In [ ]:
!pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 40.8 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "allegro/herbert-base-cased"
device = 'cuda'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

text = 'Bardzo lubię lody malinowe z bitą śmietaną.'

token_ids = tokenizer(text, return_tensors='pt')['input_ids'][0]

print ([tokenizer.decode(idx) for idx in token_ids])



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

Some weights of the model checkpoint at allegro/herbert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.sso.sso_relationship.bias', 'cls.sso.sso_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


['<s>', 'Bardzo', 'lubię', 'lody', 'mali', 'nowe', 'z', 'bi', 'tą', 'śmietan', 'ą', '.', '</s>']


In [ ]:
import random
lines = open('reviews_for_task3.txt').readlines()

random.seed(122)

def representation(L):
    txt = ' '.join(L)
    input_ids = tokenizer(txt, return_tensors='pt')['input_ids'].to(device)
    output = model(input_ids=input_ids)
    return output.last_hidden_state.detach().cpu().numpy()[0,0,:]

def spoil(L):
    res = []
    for w in L:
        if random.random() < 0.85:
            res.append(w)
        else:
            res.append(w.upper())
    return res

#lines = []

#for x in open('data/dobre.txt'):
#    lines.append('GOOD ' + x.strip())
#for x in open('data/zle.txt'):
#    lines.append('BAD ' + x.strip())

random.shuffle(lines)

N = len(lines)
test_size = N // 4
train_size = N - test_size

train_lines = lines[:train_size]
test_lines = lines[train_size:]

X_train = []
y_train = []
X_test = []
y_test = []

for line in train_lines:
    L = line.split()
    y = 0 if L[0] == 'BAD' else 1

    x = representation(L[1:])
    y_train.append(y)
    X_train.append(x)

    for i in range(3):
        x = representation(spoil(L[1:]))
        y_train.append(y)
        X_train.append(x)

    if len(X_train) % 100 == 0:
        print (len(X_train))

for line in test_lines:
    L = line.split()
    y = 0 if L[0] == 'BAD' else 1

    x = representation(L[1:])
    y_test.append(y)
    X_test.append(x)

    if len(X_test) % 100 == 0:
        print (len(X_test))

100
200
300
400
500
600
700
800
900
1000
1100
1200
100


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import svm

clf = LogisticRegression(random_state=0).fit(X_train, y_train)

print ('Train accuracy:', clf.score(X_train, y_train))
print ('Test accuracy:', clf.score(X_test, y_test))

#Train accuracy: 0.9348939283101683
#Test accuracy: 0.8715697036223929

Train accuracy: 0.9983333333333333
Test accuracy: 0.86


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math
import random

lm_name = "eryk-mazus/polka-1.1b"
lm_tokenizer = AutoTokenizer.from_pretrained(lm_name)
lm_model = AutoModelForCausalLM.from_pretrained(lm_name).to(device)
lm_model.eval()

def lm_score(text):
    enc = lm_tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = lm_model(**enc, labels=enc["input_ids"])
    loss = out.loss.item()
    ppl = math.exp(loss)
    return loss, ppl

tokenizer_config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
def spoil(L):
    res = []
    for w in L:
        if random.random() < 0.85:
            res.append(w)
        else:
            res.append(w.upper())
    return res

def bert_representation(words):
    txt = ' '.join(words)
    input_ids = tokenizer(txt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model(**input_ids)
    return output.last_hidden_state[0, 0].cpu().numpy()


In [ ]:
import torch
from torch.nn import functional as F

def conditional_continuation_logprob(prefix, continuation):
    # tokeny prefixu i pełnej sekwencji
    prefix_ids = lm_tokenizer(prefix, return_tensors='pt')['input_ids'].to(device)
    full_ids = lm_tokenizer(prefix + continuation, return_tensors='pt')['input_ids'].to(device)

    with torch.no_grad():
        out = lm_model(input_ids=full_ids)
        logits = out.logits  # [1, T, V]

    # log-proby dla tokenów 1..T-1
    logp = F.log_softmax(logits[:, :-1, :], dim=-1)
    labels = full_ids[:, 1:]  # prawdziwe następne tokeny

    token_logp = torch.gather(logp, 2, labels.unsqueeze(2)).squeeze(-1)  # [1, T-1]

    # które tokeny należą do continuation?
    prefix_len = prefix_ids.shape[1]
    # token_logp jest dla pozycji odpowiadających labels => przesunięcie o 1
    # continuation zaczyna się od tokenu o indeksie prefix_len w full_ids,
    # więc odpowiada mu logp w token_logp na indeksie prefix_len-1
    start = prefix_len - 1
    cont_logp = token_logp[0, start:].sum().item()

    return cont_logp


In [ ]:
def lm_sentiment_features(review):
    good = conditional_continuation_logprob(review, " Polecam!")
    bad  = conditional_continuation_logprob(review, " Nie polecam!")
    delta = good - bad
    return good, bad, delta


In [ ]:
import numpy as np

def full_representation(words):
    review = " ".join(words)

    bert_vec = bert_representation(words)

    good_logp, bad_logp, delta = lm_sentiment_features(review)
    lm_vec = np.array([good_logp, bad_logp, delta], dtype=float)

    return np.concatenate([bert_vec, lm_vec])


In [ ]:
#lines = open('reviews_for_task3.txt').readlines()

#random.seed(42)

N = len(lines)
test_size = N // 4
train_size = N - test_size

train_lines = lines[:train_size]
test_lines = lines[train_size:]

X_train = []
y_train = []
X_test = []
y_test = []

for line in train_lines:
    L = line.split()
    y = 0 if L[0] == 'BAD' else 1

    x = full_representation(L[1:])
    X_train.append(x)
    y_train.append(y)

    for _ in range(3):
        x = full_representation(spoil(L[1:]))
        X_train.append(x)
        y_train.append(y)

for line in test_lines:
    L = line.split()
    y = 0 if L[0] == 'BAD' else 1

    x = full_representation(L[1:])
    y_test.append(y)
    X_test.append(x)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000))
])

clf.fit(X_train, y_train)

print('Train accuracy:', clf.score(X_train, y_train))
print('Test accuracy:', clf.score(X_test, y_test))


Train accuracy: 1.0
Test accuracy: 0.85


In [ ]:
import random, numpy as np, torch

SEED = 122
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model.eval()
lm_model.eval()

lines = open('reviews_for_task3.txt', encoding='utf-8').readlines()
random.shuffle(lines)

N = len(lines)
test_size = N // 4
train_lines = lines[:-test_size]
test_lines  = lines[-test_size:]


In [ ]:
def bert_representation(words):
    txt = " ".join(words)
    inp = tokenizer(txt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inp)
    return out.last_hidden_state[0, 0].cpu().numpy()


In [ ]:
def build_Xy(lines, rep_fn, K_aug=0, aug_fn=None):
    X, y = [], []
    for line in lines:
        L = line.split()
        lab = L[0]
        words = L[1:]
        yy = 0 if lab == "BAD" else 1

        # oryginał
        X.append(rep_fn(words))
        y.append(yy)

        # augmentacje (opcjonalnie)
        if K_aug > 0 and aug_fn is not None:
            for _ in range(K_aug):
                X.append(rep_fn(aug_fn(words)))
                y.append(yy)

    return np.array(X), np.array(y)


In [ ]:
X_train0, y_train0 = build_Xy(train_lines, rep_fn=full_representation, K_aug=0)
X_test,  y_test  = build_Xy(test_lines,  rep_fn=full_representation, K_aug=0)


In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train0, y_train0)

print('Train accuracy:', clf.score(X_train0, y_train0))
print('Test accuracy:', clf.score(X_test, y_test))

Train accuracy: 0.99
Test accuracy: 0.88


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.7 MB/s eta 0:00:00


In [ ]:
import re
import random
from collections import defaultdict

def load_superbazy(path="superbazy.txt"):
    form2lemma = {}
    lemma2forms = defaultdict(list)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            form, lemma = line.split()
            form2lemma[form] = lemma
            lemma2forms[lemma].append(form)
    return form2lemma, lemma2forms

form2lemma, lemma2forms = load_superbazy("superbazy.txt")

In [ ]:
from gensim.models import Word2Vec

def tokenize_text(text):
    # proste tokeny: słowa i znaki interpunkcyjne jako osobne tokeny
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

def lemmatize_tokens(tokens):
    out = []
    for t in tokens:
        # zostaw interpunkcję
        if re.fullmatch(r"[^\w\s]", t):
            out.append(t)
            continue
        # lemat z superbazy lub lowercase jako fallback
        out.append(form2lemma.get(t, t.lower()))
    return out

# wczytaj recenzje
lines = open("reviews_for_task3.txt", encoding="utf-8").read().splitlines()
random.shuffle(lines)

N = len(lines)
test_size = N // 4
train_lines = lines[:-test_size]
test_lines  = lines[-test_size:]

texts = []
for line in train_lines:
    first_space = line.find(" ")
    review = line[first_space+1:].strip()
    toks = tokenize_text(review)
    lemmas = lemmatize_tokens(toks)
    texts.append(lemmas)

# trenuj Word2Vec
w2v = Word2Vec(
    sentences=texts,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4,
    sg=1,           # skip-gram
    negative=10,
    epochs=10
)
wv = w2v.wv


In [ ]:
STOP_LIKE = {
    "i","oraz","a","ale","że","to","w","we","na","do","z","za","od","po","nie","tak","jak",
    "się","jest","było","była","byli","były","ma","mam","mamy","są"
}

def is_word(token):
    return re.fullmatch(r"\w+", token) is not None

def is_replaceable(token):
    if not is_word(token):
        return False
    if len(token) < 4:
        return False
    if token.lower() in STOP_LIKE:
        return False
    if re.search(r"\d", token):
        return False
    return True

def best_form_by_suffix(original_form, candidate_forms):
    # próbuje dopasować końcówki od najdłuższych
    orig = original_form.lower()
    cands = candidate_forms

    # długości końcówek do sprawdzenia
    for L in [5,4,3,2,1]:
        suf = orig[-L:] if len(orig) >= L else orig
        matches = [f for f in cands if f.lower().endswith(suf)]
        if matches:
            return random.choice(matches)

    # jeśli nic nie pasuje — wybierz dowolną formę
    return random.choice(cands) if cands else None

def keep_capitalization(original, new):
    if original.isupper():
        return new.upper()
    if original[0].isupper():
        return new[0].upper() + new[1:]
    return new


In [ ]:
def replace_token_w2v(token, topn=25):
    # jeśli token nie jest w superbazie -> trudniej zachować formę -> często lepiej pominąć
    if token not in form2lemma:
        return token

    lemma = form2lemma[token]

    if lemma not in wv:
        return token

    # kandydaci-lemmaty
    candidates = [w for (w, sim) in wv.most_similar(lemma, topn=topn)]
    random.shuffle(candidates)

    for cand_lemma in candidates:
        # musi mieć formy w superbazie
        forms = lemma2forms.get(cand_lemma, [])
        if not forms:
            continue

        new_form = best_form_by_suffix(token, forms)
        if not new_form:
            continue

        new_form = keep_capitalization(token, new_form)

        # nie zwracaj identycznego
        if new_form.lower() == token.lower():
            continue

        return new_form

    return token

def augment_w2v_tokens(tokens, replace_prob=0.20):
    out = []
    for t in tokens:
        if is_replaceable(t) and random.random() < replace_prob:
            out.append(replace_token_w2v(t))
        else:
            out.append(t)
    return out


In [ ]:
X_train_aug, y_train_aug = build_Xy(train_lines, rep_fn=full_representation, K_aug=3, aug_fn=augment_w2v_tokens)


In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000)

clf.fit(X_train_aug, y_train_aug)

print('Train accuracy:', clf.score(X_train_aug, y_train_aug))
print('Test accuracy:', clf.score(X_test, y_test))

Train accuracy: 0.995
Test accuracy: 0.88
